# Comparing Normalization Techniques for the Classic CNN Reconstructor

Every reconstructor architecture in this repo -- `SimpleNet`, `Rama`, `Papyrus1stStage`/`Papyrus2ndStage` (`AI4AO/PhaseEstimators.py`), the six architectures in `CNNArchitectureComparison.ipynb`, `PupilCNN`/`PWFSNet` (`../SystemDesign/DualSensorFusion.ipynb`/`../SystemDesign/TwoStageAO.ipynb`) -- skips normalization layers entirely. This notebook holds one architecture fixed -- `CNNArchitectureComparison.ipynb`'s `ClassicCNN`: a plain, non-grouped `Conv2d` VGG-style stack directly on the 4 registered pupil channels, its depth picked from the pupil crop's actual resolution via `num_downsamples` -- and sweeps only *which normalization layer follows each conv*, training and evaluating each variant following the exact same procedure as `Tutorials/basics/05_TrainingAReconstructor.ipynb` (`AI4AO.Trainer.Trainer`, same instrument, same loss, same `TrainRunNb`). Comparing on training-loss curves and seeded closed-loop steady-state residual wavefront error (nm RMS) isolates the effect of normalization choice alone, the same controlled-ablation spirit as `CNNArchitectureComparison.ipynb`, just along a different design axis.

**The five variants**:

1. **`None`** -- no normalization layer anywhere; reproduces `ClassicCNN` exactly, conv bias included. The baseline every other variant is compared against.
2. **`BatchNorm`** -- `nn.BatchNorm2d` after every conv, batch statistics over the `AtmosParams["Nphases"]`-sized batch during training, frozen running statistics during closed-loop evaluation.
3. **`GroupNorm`** -- `nn.GroupNorm` with a fixed `num_groups=4` at every stage (28/56/112/224 channels by default all divide evenly), independent of batch size.
4. **`LayerNorm`** -- a channel-only, per-pixel normalization (ConvNeXt-style `LayerNorm2d`): each spatial location's channel vector is normalized independently. Deliberately *not* the same operation as `GroupNorm(num_groups=1, C)` (which normalizes over channels *and* space jointly per sample) -- keeping the two variants meaningfully distinct is the point of this sweep.
5. **`InstanceNorm`** -- `nn.InstanceNorm2d(affine=True)`, normalizing each channel independently per sample per spatial map (the `affine=True` override matches the other three variants' learnable scale/shift, rather than PyTorch's `affine=False` default, so all four normalized variants are equally expressive).

**Two structural choices are shared by all four normalized variants** (settled up front rather than re-derived per architecture): the conv immediately before a norm layer drops its own bias (`bias=False`, standard modern-CNN practice -- the norm's own learnable shift makes it redundant), and this notebook reuses `CNNComparison_params.py` directly (same fictional demo Pyramid + DM, no real bench, no duplicate params file) rather than copying it.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os

from AI4AO.paths import DATA_DIR
from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'


## Configuration and instrument

Reuses `CNNComparison_params.py` unchanged -- the same single fictional-demo, nominal-geometry modulated Pyramid WFS + DM (no real bench), so this comparison runs under identical atmosphere/noise/DM/loss conditions as the architecture comparison, differing only in which axis (architecture vs. normalization) is swept. Checkpoints are still kept separate, under their own `Data/NormalizationComparison/` directory, since the five models trained here are a different set of weights than the six in `CNNArchitectureComparison.ipynb`.

In [ ]:
paramfile = 'CNNComparison_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = DATA_DIR / "NormalizationComparison"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()
dm.requires_grad_(False)

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

M2C = dm.MakeZernikeM2C()
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

loss = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]
print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, Nmodes: {Nmodes}")


## The normalization layer factories

`NORM_FACTORIES` maps each variant name to either `None` (the `"None"` baseline) or a `channels -> nn.Module` factory, so `conv_stage` below can build the right normalization layer for whatever channel count a given conv stage has. `LayerNorm2d` is defined here since `nn.LayerNorm` does not operate on `(B, C, H, W)` conv feature maps directly -- it needs the channel axis moved last, matching PyTorch's own `LayerNorm2d` idiom (e.g. ConvNeXt): the permute+`nn.LayerNorm`+permute-back below normalizes each spatial location's channel vector independently, which is *not* the same operation as `GroupNorm(num_groups=1, C)` (full `(C,H,W)` per-sample normalization) -- keeping that distinction is what makes the `LayerNorm` and `GroupNorm` variants a meaningful pair to compare rather than duplicates of each other.

In [ ]:
class LayerNorm2d(nn.Module):
    """Channel-only LayerNorm for conv feature maps (ConvNeXt-style): normalizes
    each spatial location's channel vector independently. Deliberately distinct
    from GroupNorm(num_groups=1, C), which normalizes over (C,H,W) jointly."""

    def __init__(self, num_channels):
        super().__init__()
        self.norm = nn.LayerNorm(num_channels)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        return x.permute(0, 3, 1, 2)


GROUPNORM_GROUPS = 4  # fixed group count at every stage -- 28/56/112/224 channels (default base_channels=28) all divide evenly

NORM_FACTORIES = {
    "None": None,
    "BatchNorm": lambda C: nn.BatchNorm2d(C),
    "GroupNorm": lambda C: nn.GroupNorm(GROUPNORM_GROUPS, C),
    "LayerNorm": lambda C: LayerNorm2d(C),
    "InstanceNorm": lambda C: nn.InstanceNorm2d(C, affine=True),
}


## The normalized Classic CNN

`num_downsamples` and `NormalizedClassicCNN` mirror `CNNArchitectureComparison.ipynb`'s `ClassicCNN` exactly -- same depth-from-resolution logic, same `base_channels=28` default, same conv kernel sizes/activation/global-average-pool/linear head -- with normalization as the only added axis of variation, via `conv_stage`'s `norm_factory` argument. Passing `norm_factory=None` (the `"None"` variant) reproduces `ClassicCNN`'s stage exactly, conv bias included; any real `norm_factory` drops that now-redundant conv bias (`bias=False`) in favor of the norm layer's own learnable shift. Holding every other choice fixed isolates the comparison to normalization alone.

In [ ]:
def num_downsamples(size, min_size=2):
    """How many stride-2 poolings a feature map of this spatial size can
    take before dropping below min_size pixels -- same helper as
    CNNArchitectureComparison.ipynb's ClassicCNN, reused unmodified here."""
    n = 0
    while size // 2 >= min_size:
        size //= 2
        n += 1
    return n


def conv_stage(in_channels, out_channels, norm_factory, activation=nn.LeakyReLU):
    """Two 3x3 convs at out_channels, each optionally followed by a norm
    layer, then a 2x downsampling MaxPool. norm_factory=None reproduces
    ClassicCNN's stage exactly (conv bias kept); any other factory drops the
    now-redundant conv bias (bias=False) in favor of the norm layer's own
    learnable shift."""
    use_bias = norm_factory is None
    layers = [nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=use_bias)]
    if norm_factory is not None:
        layers.append(norm_factory(out_channels))
    layers.append(activation())
    layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=use_bias))
    if norm_factory is not None:
        layers.append(norm_factory(out_channels))
    layers.append(activation())
    layers.append(nn.MaxPool2d(2))
    return layers


class NormalizedClassicCNN(nn.Module):
    """ClassicCNN (CNNArchitectureComparison.ipynb) with one added axis of
    variation: which normalization layer, if any, follows each conv. Every
    other choice -- depth via num_downsamples, base_channels, conv kernel
    sizes, activation, global-average-pool + linear head -- is held fixed,
    so differences between norm_name variants isolate the effect of
    normalization alone."""

    def __init__(self, n_channels, Nmodes, Nout, norm_name, base_channels=28):
        super().__init__()

        norm_factory = NORM_FACTORIES[norm_name]
        n_stages = num_downsamples(Nout)
        channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1], norm_factory)
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)


## Instantiating all five variants

Each variant is built once here and printed with its total trainable parameter count. A normalized variant's extra learnable affine parameters (a `weight`/`bias` pair per normalized conv) roughly offset the `"None"` baseline's conv biases, so the five counts should land close together -- not because they are artificially equalized (unlike `CNNArchitectureComparison.ipynb`'s six architectures, which were explicitly sized to a shared parameter budget), but simply because normalization adds very few parameters relative to the conv weights themselves.

In [ ]:
architectures = {
    name: NormalizedClassicCNN(n_channels, Nmodes, Nout, norm_name=name).to(device=device)
    for name in NORM_FACTORIES
}

for name, model in architectures.items():
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name:12s} -- total trainable parameters: {total_params:,}")


## Training

Following `05_TrainingAReconstructor.ipynb`'s exact procedure, repeated once per normalization variant: a fresh `AdamW` optimizer and a fresh `Trainer` per variant, all sharing the same `wfs`/`dm`/`framePreprocessor`/`dataset`/`loss` objects, each calling `trainer.train(TrainRunNb, ClosedLoopIterations)`.

**A deliberate trade-off, not an oversight** (same caveat `CNNArchitectureComparison.ipynb` makes for its six architectures): `Trainer.train()` calls `self.dataset[0]` fresh at every outer step regardless of which `Trainer` instance is calling it (`PhaseDataset.__getitem__`'s `idx == 0` branch always redraws a new, statistically independent atmosphere). So training the five variants one after another on the same shared `dataset` object does **not** give them the identical per-step atmosphere realization a hand-rolled shared-draw loop would -- each variant instead trains on its own independent i.i.d. stream from the same distribution. This is fine for training itself, but it means the five *training-loss curves* below are directionally informative rather than a rigorously paired statistical test; only the seeded closed-loop rollout comparison further down is a true apples-to-apples comparison.

**One wrinkle specific to normalization**: `Trainer.train()`/`Trainer.evaluate()` already correctly call `self.phaseReconstructor.train()`/`.eval()` (confirmed in `Trainer.py`), so `BatchNorm`'s running mean/var are updated only during the `TrainRunNb` training steps and then frozen for use during the seeded closed-loop rollout below -- exactly the standard, correct behavior, requiring no extra code here. It is still worth being aware of: those running statistics are only ever averaged over `PhaseDataset`'s fresh i.i.d. draws during training (batch size `AtmosParams["Nphases"]`), so if the closed-loop rollout's r0/noise regime ever diverged from the training distribution, `BatchNorm`'s frozen statistics -- unlike `GroupNorm`/`LayerNorm`/`InstanceNorm`, which always recompute their statistics per input regardless of mode -- would not adapt to it.

Five variants of one architecture is a smaller training cost than `CNNArchitectureComparison.ipynb`'s six full architectures, but still five full training runs -- consider a smaller `TrainParams['TrainRunNb']` for an initial correctness pass before committing to the full training budget.

In [ ]:
# Closed-loop (BPTT) iterations per optimizer step in the second training stage. Stage 1 uses
# TrainParams['ClosedLoopIterations'] from CNNComparison_params.py (a single pass); this is the
# one training setting that is not in that shared file, so it lives here.
CLOSED_LOOP_ITERATIONS_STAGE2 = 10

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in architectures.items():
    print(f"=== Training {name} (stage 1: single pass) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by
        # an earlier, differently-sized version of this variant -- e.g.
        # after editing base_channels/depth above, old checkpoints under
        # PATH are no longer compatible and should be deleted or ignored.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal


### Stage 2 -- closed-loop training

The loop above (stage 1) trains every model with `TrainParams['ClosedLoopIterations']` from `CNNComparison_params.py`, i.e. a single pass per optimizer step. The next cell repeats the identical loop with `CLOSED_LOOP_ITERATIONS_STAGE2` closed-loop iterations per optimizer step, continuing from the weights stage 1 just saved (`load_checkpoint` at the top of the loop, and a fresh `AdamW` per model, as in stage 1).

The checkpoint under `PATH` is overwritten with the stage-2 weights, and from here on `trainers`/`loss_trackers`/`loss_trackers_ideal` -- the ones the plotting and evaluation cells below use -- refer to stage 2.

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in architectures.items():
    print(f"=== Training {name} (stage 2: closed loop) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by
        # an earlier, differently-sized version of this variant -- e.g.
        # after editing base_channels/depth above, old checkpoints under
        # PATH are no longer compatible and should be deleted or ignored.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], CLOSED_LOOP_ITERATIONS_STAGE2)

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal


## Comparing training-loss curves

`Trainer.plot_losses` only plots one tracker pair at a time, so we reimplement just its smoothing-and-overlay logic here for all five trackers at once. The oracle "ideal loss" bound (`z_inv` applied to the true residual OPD) is a property of the atmosphere/noise distribution and the DM, not of the normalization choice, so the five `loss_tracker_ideal` curves are statistically equivalent -- only one is shown, as a reference lower bound shared by all five.

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")

fig, ax = plt.subplots(figsize=(8, 5))
for name, tracker in loss_trackers.items():
    ax.plot(smooth(tracker), label=name)

first_ideal = next(iter(loss_trackers_ideal.values()))
ax.plot(smooth(first_ideal), label="Oracle bound (ideal)", color="black", linestyle="--")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"Loss")
ax.legend()
plt.show()


## Comparing closed-loop residual wavefront error (nm RMS)

For each variant, reseed immediately before the rollout (`../SystemDesign/DualSensorFusion.ipynb`'s "reset the seed right before each rollout" trick) so all five see identical wind/r0/noise draws, then run `trainer.evaluate()` and compute the steady-state residual wavefront error in nm RMS over the pupil, excluding the leaky-integrator warm-up (`i > n_steps * 0.3`, the same convention `Trainer.evaluate()` uses internally).

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator
    warm-up period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {}
for name, trainer in trainers.items():
    torch.manual_seed(TrainParams['Seed'])
    result = trainer.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    print(f"{name:12s} -- steady-state residual: {nm_rms[name]:.1f} nm RMS")


In [ ]:
names = list(nm_rms.keys())
rms_values = [nm_rms[n] for n in names]
param_counts = [sum(p.numel() for p in architectures[n].parameters() if p.requires_grad) for n in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(names, rms_values)
axes[0].set_ylabel("Steady-state residual (nm RMS)")
axes[0].set_title("Closed-loop performance")
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(names, param_counts)
axes[1].set_ylabel("Trainable parameters")
axes[1].set_title("Model size")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## Visualizing the best normalization variant's closed loop

A closer look at whichever variant came out on top above -- the same rollout-animation pattern `05_TrainingAReconstructor.ipynb`/`CNNArchitectureComparison.ipynb` use.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

best_name = min(nm_rms, key=nm_rms.get)
print(f"Best normalization variant: {best_name} ({nm_rms[best_name]:.1f} nm RMS)")

n_frames = 60
torch.manual_seed(TrainParams['Seed'])
result = trainers[best_name].evaluate(n_steps=n_frames, dataset=eval_dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number = 9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes,
        max_channel_number = 9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())
